In [1]:
# 3.1 Import Library
import pandas as pd
import numpy as np
import os
import ast
import json
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [2]:
# 3.2 Konfigurasi Path
LEXICON_PATH = '../../kamus/sla_lexicon_adapted.txt' 
INPUT_FILE   = '../outputs/lexicon_matching.csv'
OUTPUT_DIR   = '../outputs/'
OUTPUT_FILE  = os.path.join(OUTPUT_DIR, 'vader_heuristics.csv')
STATS_FILE   = os.path.join(OUTPUT_DIR, 'vader_stats.json')
CLEAN_OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'sentiment_clean.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# 3.3 Load Data dan Lexicon
def load_custom_lexicon(filepath):
    lexicon = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                word = parts[0]
                mean_score = float(parts[1])
                lexicon[word] = mean_score
    return lexicon

# Load Lexicon
print(f"[INFO] Memuat leksikon adaptasi dari: {LEXICON_PATH}")
sla_dict = load_custom_lexicon(LEXICON_PATH)
print(f"[INFO] Berhasil memuat {len(sla_dict)} entri leksikon.")

# Load Data Matching
df = pd.read_csv(INPUT_FILE)

# Convert kolom list string -> list python
list_cols = ['tokens', 'matched_words', 'ignored_words', 'unmatched_words']
for col in list_cols:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Definisi Ignore Set (lowercase untuk konsistensi)
KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak', 'secara', 'antara'
}
PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu',
    'yang', 'mereka', 'kita'
}
PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa',
    'bagaimana', 'kan', 'dong'
}
KATA_BENDA_NETRAL = {
    'rapat', 'sidang', 'pemerintah', 'komisi', 'fraksi', 'partai',
    'indonesia', 'daerah', 'keuangan', 'persidangan', 'kebijakan',
    'program', 'proyek', 'anggaran', 'undang', 'pasal', 'ayat',
    'ketua', 'anggota', 'wakil', 'menteri', 'presiden', 'dpr', 'ri', 'ruu'
}

IGNORE_CATEGORIES = (
    KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
    | KATA_BENDA_NETRAL
)

# Working lexicon: kata di ignore dipaksa skor 0.0
working_lexicon = sla_dict.copy()
for word in IGNORE_CATEGORIES:
    working_lexicon[word] = 0.0

# Rekonstruksi Sequence 
def reconstruct_sequence_ignore(tokens, lexicon, ignore_set):
    seq_tokens = []
    seq_scores = []
    for tok in tokens:
        t_low = tok.lower()
        seq_tokens.append(tok)
        if t_low in lexicon:
            score = 0.0 if t_low in ignore_set else lexicon[t_low]
            seq_scores.append({'word': tok, 'score': score, 'matched': True})
        else:
            seq_scores.append({'word': tok, 'score': 0.0, 'matched': False})
    return seq_tokens, seq_scores

# Apply ke dataframe
df[['filtered_tokens', 'filtered_word_scores']] = df.apply(
    lambda row: pd.Series(
        reconstruct_sequence_ignore(row['tokens'], working_lexicon, IGNORE_CATEGORIES)
    ),
    axis=1
)

print(f"[INFO] Data loaded: {len(df)} tweets")
print(f"[INFO] Ignore Function & Lexicon Applied.")

[INFO] Memuat leksikon adaptasi dari: ../../kamus/sla_lexicon_adapted.txt
[INFO] Berhasil memuat 9855 entri leksikon.
[INFO] Data loaded: 13192 tweets
[INFO] Ignore Function & Lexicon Applied.


In [4]:
# 3.4 Parameter Heuristik VADER (REVISI)

# Punctuation
PUNCT_BOOST = 0.292
MAX_EXCLAMATION = 4

# Capitalization
CAP_BOOST = 0.733

# Booster / Dampener
B_INCR = 0.293
B_DECR = -0.293

BOOSTERS = {
    'sangat': B_INCR,
    'sekali': B_INCR,
    'banget': B_INCR,
    'amat': B_INCR,
    'teramat': B_INCR,
    'terlalu': B_INCR,
    'benar': B_INCR,
    'betul': B_INCR,
    'pasti': B_INCR,
    'jelas': B_INCR,
    'nyata': B_INCR,
    'parah': B_INCR,

    'agak': B_DECR,
    'sedikit': B_DECR,
    'kurang': B_DECR,
    'hampir': B_DECR,
    'nyaris': B_DECR,
    'hanya': B_DECR,
    'cuma': B_DECR,
    'sekadar': B_DECR,
    'mungkin': B_DECR,
    'lumayan': B_DECR,
    'cukup': B_DECR
}

# Contrastive conjunction
CONTRAST_CONJUNCTIONS = [
    'tapi', 'tetapi', 'namun',
    'walaupun', 'meskipun',
    'kendati', 'sedangkan',
    'padahal'
]

# Negation
NEGATIONS = [
    'tidak', 'tak', 'nggak',
    'gak', 'bukan', 'jangan',
    'tanpa', 'belum', 'never',
    'no'
]

NEGATION_WINDOW = 3
NEGATION_SCALAR = -0.74

# Normalization
ALPHA = 15.0

print("[INFO] Heuristic Parameters loaded.")

[INFO] Heuristic Parameters loaded.


In [5]:
# 3.5 Fungsi Heuristik VADER 
# Parameter Utama
PUNCT_BOOST     = 0.292
MAX_PUNCT_BOOST = 1.168
CAP_BOOST       = 0.733

B_INCR = 0.293
B_DECR = -0.293

NEGATION_SCALAR = -0.74
NEGATION_WINDOW = 3

ALPHA = 15.0


# Booster / Degree Modifiers
BOOSTERS = {
    'sangat': B_INCR,
    'sekali': B_INCR,
    'banget': B_INCR,
    'amat': B_INCR,
    'teramat': B_INCR,
    'terlalu': B_INCR,
    'benar': B_INCR,
    'betul': B_INCR,
    'pasti': B_INCR,
    'jelas': B_INCR,
    'nyata': B_INCR,
    'parah': B_INCR,
    'luar': B_INCR,
    'luar biasa': B_INCR,

    'agak': B_DECR,
    'sedikit': B_DECR,
    'kurang': B_DECR,
    'hampir': B_DECR,
    'nyaris': B_DECR,
    'hanya': B_DECR,
    'cuma': B_DECR,
    'sekadar': B_DECR,
    'mungkin': B_DECR,
    'lumayan': B_DECR,
    'cukup': B_DECR
}


# Contrastive Conjunctions
CONTRAST_CONJUNCTIONS = [
    'tapi', 'tetapi', 'namun', 'walaupun',
    'meskipun', 'kendati', 'sekalipun',
    'sedangkan', 'padahal', 'sementara',
    'biarpun', 'walau', 'meski'
]


# Negation Words
NEGATIONS = [
    'tidak', 'tak', 'nggak', 'gak',
    'bukan', 'jangan', 'tanpa',
    'belum', 'no', 'never'
]

print("[INFO] Heuristic parameters loaded.")

[INFO] Heuristic parameters loaded.


In [6]:
# 3.5.1 Punctuation Heuristic

def apply_punctuation_heuristic(text, base_score):

    if not isinstance(text, str):
        return base_score, {
            'exclamation_count': 0,
            'question_count': 0,
            'punct_boost': 0
        }

    exclamation_count = text.count('!')
    question_count = text.count('?')

    # Exclamation emphasis
    ep_amplifier = min(exclamation_count, 4) * 0.292

    # Question mark emphasis
    qm_amplifier = 0

    if question_count > 1:
        if question_count <= 3:
            qm_amplifier = question_count * 0.18
        else:
            qm_amplifier = 0.96

    punct_boost = ep_amplifier + qm_amplifier

    if base_score > 0:
        final_score = base_score + punct_boost
    elif base_score < 0:
        final_score = base_score - punct_boost
    else:
        final_score = base_score

    return final_score, {
        'exclamation_count': exclamation_count,
        'question_count': question_count,
        'punct_boost': punct_boost
    }

print("[INFO] Punctuation heuristic ready.")

[INFO] Punctuation heuristic ready.


In [7]:
# 3.5.2 Capitalization Heuristic

def apply_capitalization_heuristic(tokens, word_scores):

    sentiments = []

    allcaps_words = [t for t in tokens if t.isupper()]
    is_cap_diff = 0 < len(allcaps_words) < len(tokens)

    for i, token_data in enumerate(word_scores):

        score = token_data['score']
        token = tokens[i]

        if score != 0 and token.isupper() and is_cap_diff:

            if score > 0:
                score += CAP_BOOST
            else:
                score -= CAP_BOOST

        sentiments.append(score)

    adjusted_score = sum(sentiments)

    return adjusted_score, {
        'cap_diff': is_cap_diff,
        'caps_words': len(allcaps_words)
    }

print("[INFO] Capitalization heuristic ready.")

[INFO] Capitalization heuristic ready.


In [8]:
# 3.5.3 Degree Modifier Heuristic

def apply_degree_modifier_heuristic(tokens, word_scores):

    adjusted_scores = []
    adjustments = []

    for i, token_data in enumerate(word_scores):

        score = token_data['score']

        if score == 0:
            adjusted_scores.append(score)
            continue

        modified_score = score

        # Cek 3 token sebelumnya
        for distance in range(1, 4):

            prev_idx = i - distance

            if prev_idx < 0:
                continue

            prev_token = tokens[prev_idx].lower()

            if prev_token in BOOSTERS:

                scalar = BOOSTERS[prev_token]

                # Polarity-sensitive
                if modified_score < 0:
                    scalar *= -1

                # Distance decay
                if distance == 2:
                    scalar *= 0.95
                elif distance == 3:
                    scalar *= 0.9

                modified_score += scalar

                adjustments.append({
                    'modifier': prev_token,
                    'affected_word': tokens[i],
                    'scalar': scalar
                })

        adjusted_scores.append(modified_score)

    return sum(adjusted_scores), adjustments

print("[INFO] Degree modifier heuristic ready.")

[INFO] Degree modifier heuristic ready.


In [9]:
# 3.5.4 Negation Heuristic

def apply_negation_heuristic(tokens, word_scores):

    adjusted_scores = []
    negations_found = []

    for i, token_data in enumerate(word_scores):

        score = token_data['score']

        if score == 0:
            adjusted_scores.append(score)
            continue

        modified_score = score

        # Cek 3 token sebelumnya
        for distance in range(1, NEGATION_WINDOW + 1):

            prev_idx = i - distance

            if prev_idx < 0:
                continue

            prev_token = tokens[prev_idx].lower()

            if prev_token in NEGATIONS:

                modified_score *= NEGATION_SCALAR

                negations_found.append({
                    'negation': prev_token,
                    'affected_word': tokens[i],
                    'distance': distance
                })

                break

        adjusted_scores.append(modified_score)

    return sum(adjusted_scores), negations_found

print("[INFO] Negation heuristic ready.")

[INFO] Negation heuristic ready.


In [10]:
# 3.5.5 Contrastive Conjunction Heuristic

def apply_polarity_shift_heuristic(tokens, word_scores):

    sentiments = [w['score'] for w in word_scores]
    lowered_tokens = [t.lower() for t in tokens]

    contrast_positions = [
        i for i, t in enumerate(lowered_tokens)
        if t in CONTRAST_CONJUNCTIONS
    ]

    if not contrast_positions:
        return sum(sentiments), {
            'contrast_found': False
        }

    but_idx = contrast_positions[0]

    adjusted_sentiments = []

    for i, score in enumerate(sentiments):

        if i < but_idx:
            adjusted_sentiments.append(score * 0.5)

        elif i > but_idx:
            adjusted_sentiments.append(score * 1.5)

        else:
            adjusted_sentiments.append(score)

    return sum(adjusted_sentiments), {
        'contrast_found': True,
        'contrast_word': tokens[but_idx],
        'contrast_position': but_idx
    }

print("[INFO] Contrastive conjunction heuristic ready.")

[INFO] Contrastive conjunction heuristic ready.


In [11]:
# 3.6 Implementasi Scoring (VERSI SEQUENTIAL / BERURUTAN)

print("\n[PROSES] Menjalankan Scoring SLA + Heuristics (Sequential Pipeline)...")
results = []

for idx, row in df.iterrows():
    if (idx + 1) % 1000 == 0:
        print(f"Processing tweet {idx + 1}/{len(df)}")

    text = row['teks']
    tokens = row['filtered_tokens']
    word_scores = row['filtered_word_scores']  # List of dict: {'word': tok, 'score': score, 'matched': True/False}
    
    # 1. BASE SCORE: Jumlah skor leksikon yang matched
    current_scores = [w['score'] for w in word_scores]
    
    # 2. DEGREE MODIFIERS (Boosters dan Dampeners)
    modified_scores = current_scores.copy()
    for i in range(len(modified_scores)):
        if modified_scores[i] == 0: 
            continue
        for dist in range(1, 4):
            prev_idx = i - dist
            if prev_idx < 0: continue
            prev_tok = tokens[prev_idx].lower()
            if prev_tok in BOOSTERS:
                scalar = BOOSTERS[prev_tok]
                if modified_scores[i] < 0: scalar *= -1  # Polarity-sensitive
                if dist == 2: scalar *= 0.95
                elif dist == 3: scalar *= 0.90
                modified_scores[i] += scalar

    # 3. NEGATION SCALING
    negated_scores = modified_scores.copy()
    for i in range(len(negated_scores)):
        if negated_scores[i] == 0: 
            continue
        for dist in range(1, NEGATION_WINDOW + 1):
            prev_idx = i - dist
            if prev_idx < 0: continue
            prev_tok = tokens[prev_idx].lower()
            if prev_tok in NEGATIONS:
                negated_scores[i] *= NEGATION_SCALAR
                break  # Hanya 1 negasi terdekat yang berlaku per kata

    # 4. CONTRASTIVE CONJUNCTION SHIFT
    lowered_tokens = [t.lower() for t in tokens]
    contrast_idx = next((i for i, t in enumerate(lowered_tokens) if t in CONTRAST_CONJUNCTIONS), None)
    contrast_scores = negated_scores.copy()
    if contrast_idx is not None:
        contrast_scores = [
            s * 0.5 if i < contrast_idx else 
            s * 1.5 if i > contrast_idx else s 
            for i, s in enumerate(contrast_scores)
        ]

    # 5. SUM TOTAL SCORE
    total_score = sum(contrast_scores)

    # 6. PUNCTUATION BOOST/DAMPEN (Applied to total sum)
    punct_adjusted, _ = apply_punctuation_heuristic(text, total_score)
    
    # 7. CAPITALIZATION BOOST (Applied to total sum)
    # VADER original: cap boost applies if ALL CAPS or mixed case exists
    all_caps_ratio = sum(1 for t in tokens if t.isupper() and t.isalpha()) / max(len(tokens), 1)
    if 0.15 <= all_caps_ratio <= 0.85 and total_score != 0:
        cap_adjusted = punct_adjusted + (CAP_BOOST if total_score > 0 else -CAP_BOOST)
    else:
        cap_adjusted = punct_adjusted

    # 8. NORMALIZATION (VADER Formula)
    compound_score = cap_adjusted / np.sqrt(cap_adjusted**2 + ALPHA)
    compound_score = float(np.clip(compound_score, -1.0, 1.0))
    
    # Hitung rasio pos, neg, neu dari skor akhir per kata
    final_scores = contrast_scores
    pos_sum = sum(max(0, s) for s in final_scores)
    neg_sum = sum(abs(min(0, s)) for s in final_scores)
    total_valence = pos_sum + neg_sum
    
    if total_valence > 0:
        pos_ratio = round(pos_sum / total_valence, 4)
        neg_ratio = round(neg_sum / total_valence, 4)
    else:
        pos_ratio = 0.0
        neg_ratio = 0.0
    neu_ratio = round(1.0 - pos_ratio - neg_ratio, 4)
    
    # membulatkan compound_score menjadi 4 desimal
    compound_score = round(compound_score, 4)

    # 9. THRESHOLD CLASSIFICATION
    if compound_score >= 0.05:
        sentiment = 'positive'
    elif compound_score <= -0.05:
        sentiment = 'negative'
    else:
        sentiment = 'neutral'

    results.append({
        'no': row['no'],
        'timestamp': row.get('timestamp', ''),
        'teks': text,
        'teks_processed': row.get('teks_processed', ''),
        'base_score': total_score,
        'final_raw_score': cap_adjusted,
        'compound_score': compound_score,
        'sentiment': sentiment,
        'pos': pos_ratio,
        'neg': neg_ratio,
        'neu': neu_ratio
    })

print(f"\n[SELESAI] Memproses {len(results)} tweet.")


[PROSES] Menjalankan Scoring SLA + Heuristics (Sequential Pipeline)...
Processing tweet 1000/13192
Processing tweet 2000/13192
Processing tweet 3000/13192
Processing tweet 4000/13192
Processing tweet 5000/13192
Processing tweet 6000/13192
Processing tweet 7000/13192
Processing tweet 8000/13192
Processing tweet 9000/13192
Processing tweet 10000/13192
Processing tweet 11000/13192
Processing tweet 12000/13192
Processing tweet 13000/13192

[SELESAI] Memproses 13192 tweet.


In [12]:
# 3.7 Statistik
df_results = pd.DataFrame(results)

stats = {
    'total_tweets': len(df_results),
    'sentiment_distribution': df_results['sentiment'].value_counts().to_dict(),
    'compound_score_stats': {
        'mean': float(df_results['compound_score'].mean()),
        'std': float(df_results['compound_score'].std()),
        'min': float(df_results['compound_score'].min()),
        'max': float(df_results['compound_score'].max())
    }
}

print("\n[STATISTIK] Distribusi Sentimen:")
for sent, count in stats['sentiment_distribution'].items():
    print(f"  {sent}: {count} ({count/len(df_results)*100:.2f}%)")

print("\n[PREVIEW] 5 Data Pertama:")
display(df_results[['no', 'teks_processed', 'compound_score', 'sentiment']].head())


[STATISTIK] Distribusi Sentimen:
  positive: 6712 (50.88%)
  negative: 4854 (36.80%)
  neutral: 1626 (12.33%)

[PREVIEW] 5 Data Pertama:


,no,teks_processed,compound_score,sentiment
0,1,ADIL loh untuk yang punya kebijakan publik neg...,0.6636,positive
1,2,tertibkan media online DPR pemerintah jangan s...,-0.8627,negative
2,3,harus dievaluasi lagi kebijakan bebas visa ter...,-0.8745,negative
3,4,jangan ngambang aturan logis apa undang undang,-0.5267,negative
4,5,kebebasan bersuara berpendapat memang dijamin ...,-0.6553,negative


In [13]:
# 3.8 Simpan Hasil ke CSV dan JSON
df.to_csv(OUTPUT_FILE, index=False)
print(f"[INFO] Hasil utama disimpan: {OUTPUT_FILE}")

with open(STATS_FILE, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f"[INFO] Statistik disimpan: {STATS_FILE}")

[INFO] Hasil utama disimpan: ../outputs/vader_heuristics.csv
[INFO] Statistik disimpan: ../outputs/vader_stats.json


In [14]:
# 3.9 Simpan Versi Ringkas
df_clean = df_results[['no', 'timestamp', 'teks', 'teks_processed', 'sentiment', 'compound_score']]
df_clean.to_csv(CLEAN_OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n[INFO] Hasil tersimpan di:")
print(f"  - Full   : {OUTPUT_FILE}")
print(f"  - Stats  : {STATS_FILE}")
print(f"  - Clean  : {CLEAN_OUTPUT_FILE}")


[INFO] Hasil tersimpan di:
  - Full   : ../outputs/vader_heuristics.csv
  - Stats  : ../outputs/vader_stats.json
  - Clean  : ../outputs/sentiment_clean.csv


In [17]:
# 3.10 Export Format Standar VADER
print("\n[FORMAT] Menyimpan output format standar VADER...")

# Buat kolom dictionary dengan urutan key sesuai permintaan
df_vader_format = df_results.copy()
df_vader_format['vader_output'] = df_vader_format.apply(
    lambda row: str({
        'pos': row['pos'],
        'compound': row['compound_score'],
        'neu': row['neu'],
        'neg': row['neg']
    }), axis=1
)

# Simpan ke CSV
output_vader_csv = os.path.join(OUTPUT_DIR, 'vader_standard_output.csv')
df_vader_format[['teks', 'teks_processed', 'vader_output']].to_csv(output_vader_csv, index=False, encoding='utf-8')

# Tampilkan preview 5 baris
print("\n[PREVIEW] 5 Baris Pertama")
for _, row in df_vader_format.head().iterrows():
    # Potong teks jika terlalu panjang agar rapi di console
    text_display = row['teks_processed'][:60] + "..." if len(row['teks']) > 60 else row['teks']
    print(f"{text_display} ----------------------------- {row['vader_output']}")

print(f"\n[INFO] File format VADER tersimpan: {output_vader_csv}")


[FORMAT] Menyimpan output format standar VADER...

[PREVIEW] 5 Baris Pertama
ADIL loh untuk yang punya kebijakan publik negara ingat yang... ----------------------------- {'pos': 1.0, 'compound': 0.6636, 'neu': 0.0, 'neg': 0.0}
tertibkan media online DPR pemerintah jangan sporadis apalag... ----------------------------- {'pos': 0.1748, 'compound': -0.8627, 'neu': -0.0, 'neg': 0.8252}
harus dievaluasi lagi kebijakan bebas visa terutama untuk ne... ----------------------------- {'pos': 0.3462, 'compound': -0.8745, 'neu': -0.0, 'neg': 0.6538}
jangan ngambang aturan logis apa undang undang ----------------------------- {'pos': 0.0, 'compound': -0.5267, 'neu': 0.0, 'neg': 1.0}
kebebasan bersuara berpendapat memang dijamin UU tetapi kebe... ----------------------------- {'pos': 0.3939, 'compound': -0.6553, 'neu': 0.0, 'neg': 0.6061}

[INFO] File format VADER tersimpan: ../outputs/vader_standard_output.csv
